# Image Classification Preset

Reusable Google Colab notebook template for a Kaggle-style image classification competition.

## Usage guide

1. Run the install and import cells.
2. Edit the **Configuration** cell. The inferred defaults are based on the inspected dataset, but every important path and column name is editable.
3. Upload the dataset zip in Colab, mount Google Drive, or point `ZIP_PATH` / `DATA_DIR` to existing files.
4. Run cells top to bottom.
5. Check validation accuracy, then generate `submission.csv`.

## Inferred task summary

- Likely task type: binary image classification.
- Training files: image folder plus annotation CSV.
- Test files: image folder plus sample submission CSV.
- Observed annotation columns: `image_name` and `class`.
- Observed sample submission columns: `id` and `answer`.
- Likely leaderboard metric from the overview: Accuracy Score.

This notebook intentionally avoids hard-coding a competition name. Treat the configuration cell as the single place to adjust dataset details.

## 1. Install Required Libraries

Colab usually includes PyTorch, torchvision, pandas, scikit-learn, Pillow, and matplotlib. This cell installs missing lightweight packages only when needed.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "torchvision": "torchvision",
}

missing = [pip_name for import_name, pip_name in required_packages.items()
           if importlib.util.find_spec(import_name) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available.")

# Optional advanced models:
# !pip -q install timm

## 2. Imports

In [ ]:
from pathlib import Path
import os
import random
import zipfile
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

try:
    from google.colab import files, drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 3. Configuration Placeholders

Edit this cell first. Inferred values are grouped here so you can quickly adapt the notebook if Kaggle changes file names, columns, or folder structure.

In [ ]:
# -----------------------------
# Competition / task settings
# -----------------------------
TASK_TYPE = "image_classification"
LIKELY_METRIC = "accuracy"
RANDOM_STATE = 42

# -----------------------------
# Data locations
# -----------------------------
# In Colab, either:
# - leave ZIP_PATH = None and upload the zip when prompted,
# - set ZIP_PATH to a file in /content or Google Drive,
# - or set DATA_DIR to a folder where the data is already extracted.
DATA_DIR = Path("/content/data")
ZIP_PATH = None  # TODO: example: Path("/content/dataset.zip") or Path("/content/drive/MyDrive/dataset.zip")
USE_GOOGLE_DRIVE = False
FORCE_EXTRACT = False

# -----------------------------
# Inferred dataset file names
# -----------------------------
TRAIN_PATH = None  # TODO: keep None for auto-detect, or set Path("/content/data/train.csv")
TEST_PATH = None   # Optional CSV. This dataset appears to use test images + sample_submission.csv.
SAMPLE_SUBMISSION_PATH = None  # TODO: keep None for auto-detect, or set explicitly.

# -----------------------------
# Inferred image folders
# -----------------------------
# Keep None to search recursively under DATA_DIR.
# Inspected zip used nested folders like train/train/*.jpg and test/test/*.jpg.
TRAIN_IMAGE_DIR = None
TEST_IMAGE_DIR = None
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# -----------------------------
# Inferred columns
# -----------------------------
IMAGE_COLUMN = "image_name"       # TODO: training image filename column
TARGET_COLUMN = "class"           # TODO: training label column
ID_COLUMN = "id"                  # TODO: sample submission id column
SUBMISSION_TARGET_COLUMN = "answer"  # TODO: sample submission prediction column

# -----------------------------
# Modeling settings
# -----------------------------
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VALID_SIZE = 0.2
NUM_WORKERS = 2
USE_PRETRAINED = True
FREEZE_BACKBONE = True

# -----------------------------
# Output
# -----------------------------
OUTPUT_PATH = Path("/content/submission.csv")

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

## 4. Dataset Extraction And File Inspection

In [ ]:
DATA_DIR = Path(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

if USE_GOOGLE_DRIVE and IN_COLAB:
    drive.mount("/content/drive")

def has_files(path):
    return path.exists() and any(path.iterdir())

if ZIP_PATH is None and not has_files(DATA_DIR):
    if IN_COLAB:
        print("Upload the dataset zip file.")
        uploaded = files.upload()
        zip_candidates = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
        ZIP_PATH = Path(zip_candidates[0] if zip_candidates else next(iter(uploaded.keys())))
    else:
        print("Not running in Colab. Set ZIP_PATH or place extracted files in DATA_DIR.")

if ZIP_PATH is not None:
    ZIP_PATH = Path(ZIP_PATH)
    should_extract = FORCE_EXTRACT or not has_files(DATA_DIR)
    print("ZIP_PATH:", ZIP_PATH)
    print("DATA_DIR:", DATA_DIR)
    if should_extract:
        print("Extracting dataset...")
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(DATA_DIR)
        print("Extraction complete.")
    else:
        print("DATA_DIR already contains files. Set FORCE_EXTRACT=True to extract again.")

def print_tree(root, max_items=120):
    root = Path(root)
    print(f"\nFile preview under: {root}")
    if not root.exists():
        print("Path does not exist.")
        return
    items = sorted(root.rglob("*"))
    for path in items[:max_items]:
        rel = path.relative_to(root)
        suffix = "/" if path.is_dir() else ""
        print(f"  {rel}{suffix}")
    if len(items) > max_items:
        print(f"  ... ({len(items) - max_items} more items)")

def count_extensions(root):
    counts = Counter()
    for path in Path(root).rglob("*"):
        if path.is_file():
            counts[path.suffix.lower() or "<no_ext>"] += 1
    return counts

print_tree(DATA_DIR, max_items=80)
print("\nExtension counts:")
for ext, count in count_extensions(DATA_DIR).most_common():
    print(f"{ext}: {count}")

## 5. Load Data

In [ ]:
def find_first_by_name(root, names):
    root = Path(root)
    names_lower = {name.lower() for name in names}
    matches = [p for p in root.rglob("*") if p.is_file() and p.name.lower() in names_lower]
    matches = sorted(matches, key=lambda p: (len(p.parts), str(p).lower()))
    return matches[0] if matches else None

def list_image_files(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])

TRAIN_PATH = Path(TRAIN_PATH) if TRAIN_PATH else find_first_by_name(DATA_DIR, ["train.csv"])
SAMPLE_SUBMISSION_PATH = (
    Path(SAMPLE_SUBMISSION_PATH)
    if SAMPLE_SUBMISSION_PATH
    else find_first_by_name(DATA_DIR, ["sample_submission.csv"])
)
TEST_PATH = Path(TEST_PATH) if TEST_PATH else find_first_by_name(DATA_DIR, ["test.csv"])

print("Detected TRAIN_PATH:", TRAIN_PATH)
print("Detected TEST_PATH:", TEST_PATH)
print("Detected SAMPLE_SUBMISSION_PATH:", SAMPLE_SUBMISSION_PATH)

if TRAIN_PATH is None:
    raise FileNotFoundError("Could not find train.csv. Set TRAIN_PATH in the configuration cell.")
if SAMPLE_SUBMISSION_PATH is None:
    raise FileNotFoundError("Could not find sample_submission.csv. Set SAMPLE_SUBMISSION_PATH in the configuration cell.")

train_df = pd.read_csv(TRAIN_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
test_df = pd.read_csv(TEST_PATH) if TEST_PATH is not None else sample_submission.copy()

print("\nRaw shapes")
print("train_df:", train_df.shape)
print("test_df:", test_df.shape)
print("sample_submission:", sample_submission.shape)

print("\nTrain columns:", train_df.columns.tolist())
print("Sample submission columns:", sample_submission.columns.tolist())

display(train_df.head())
display(sample_submission.head())

## 6. Basic Data Overview

In [ ]:
print("Missing values in train_df:")
display(train_df.isna().sum().to_frame("missing"))

print("\nMissing values in sample_submission:")
display(sample_submission.isna().sum().to_frame("missing"))

if TARGET_COLUMN not in train_df.columns:
    raise ValueError(f"TARGET_COLUMN={TARGET_COLUMN!r} is not in train_df. Edit the configuration cell.")
if IMAGE_COLUMN not in train_df.columns:
    raise ValueError(f"IMAGE_COLUMN={IMAGE_COLUMN!r} is not in train_df. Edit the configuration cell.")
if ID_COLUMN not in sample_submission.columns:
    raise ValueError(f"ID_COLUMN={ID_COLUMN!r} is not in sample_submission. Edit the configuration cell.")
if SUBMISSION_TARGET_COLUMN not in sample_submission.columns:
    raise ValueError(
        f"SUBMISSION_TARGET_COLUMN={SUBMISSION_TARGET_COLUMN!r} is not in sample_submission. "
        "Edit the configuration cell."
    )

label_counts = train_df[TARGET_COLUMN].value_counts(dropna=False).sort_index()
print("Target distribution:")
display(label_counts.to_frame("count"))

num_classes = train_df[TARGET_COLUMN].nunique(dropna=True)
print("Number of classes:", num_classes)
print("Likely metric:", LIKELY_METRIC)

## 7. Basic EDA

In [ ]:
plt.figure(figsize=(6, 4))
train_df[TARGET_COLUMN].value_counts().sort_index().plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel(TARGET_COLUMN)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

train_image_root = Path(TRAIN_IMAGE_DIR) if TRAIN_IMAGE_DIR else DATA_DIR
test_image_root = Path(TEST_IMAGE_DIR) if TEST_IMAGE_DIR else DATA_DIR

all_train_search_images = list_image_files(train_image_root)
all_test_search_images = list_image_files(test_image_root)

print(f"Images found under train_image_root ({train_image_root}): {len(all_train_search_images)}")
print(f"Images found under test_image_root ({test_image_root}): {len(all_test_search_images)}")

name_to_image_path = {}
for path in all_train_search_images:
    # Keep the first path for each filename. If filenames collide, set TRAIN_IMAGE_DIR/TEST_IMAGE_DIR manually.
    name_to_image_path.setdefault(path.name, path)

train_df = train_df.copy()
train_df["image_path"] = train_df[IMAGE_COLUMN].astype(str).map(name_to_image_path)

missing_train_images = train_df["image_path"].isna().sum()
print("Training rows without matched image file:", missing_train_images)
if missing_train_images:
    display(train_df.loc[train_df["image_path"].isna()].head())

sample_rows = train_df.dropna(subset=["image_path"]).sample(
    n=min(12, train_df["image_path"].notna().sum()),
    random_state=RANDOM_STATE,
)

cols = 4
rows = int(np.ceil(len(sample_rows) / cols))
plt.figure(figsize=(cols * 3, rows * 3))
for i, (_, row) in enumerate(sample_rows.iterrows(), start=1):
    img = Image.open(row["image_path"]).convert("RGB")
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.title(f"{TARGET_COLUMN}: {row[TARGET_COLUMN]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Preprocessing

In [ ]:
train_df = train_df.dropna(subset=["image_path", TARGET_COLUMN]).reset_index(drop=True)

label_values = sorted(train_df[TARGET_COLUMN].dropna().unique().tolist())
label_to_idx = {label: idx for idx, label in enumerate(label_values)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

train_df["label_idx"] = train_df[TARGET_COLUMN].map(label_to_idx).astype(int)

print("Label mapping:")
print(label_to_idx)

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

valid_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class ImageClassificationDataset(Dataset):
    def __init__(self, df, image_path_col="image_path", label_col=None, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_path_col = image_path_col
        self.label_col = label_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row[self.image_path_col]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        if self.label_col is None:
            return image
        label = torch.tensor(int(row[self.label_col]), dtype=torch.long)
        return image, label

## 9. Feature Engineering

In [ ]:
# Image models learn directly from pixels, so the safe baseline uses no handcrafted features.
# Optional quick metadata checks can still help debug duplicates or leakage.

train_df["filename_stem"] = train_df[IMAGE_COLUMN].astype(str).map(lambda x: Path(x).stem)
train_df["filename_length"] = train_df[IMAGE_COLUMN].astype(str).str.len()

print("Duplicate training filenames:", train_df[IMAGE_COLUMN].duplicated().sum())
print("Duplicate filename stems:", train_df["filename_stem"].duplicated().sum())
display(train_df[[IMAGE_COLUMN, TARGET_COLUMN, "filename_length"]].head())

## 10. Train / Validation Split

In [ ]:
class_counts = train_df["label_idx"].value_counts()
can_stratify = class_counts.min() >= 2 and len(class_counts) > 1
stratify_values = train_df["label_idx"] if can_stratify else None

train_part, valid_part = train_test_split(
    train_df,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_values,
)

print("Train split:", train_part.shape)
print("Valid split:", valid_part.shape)
print("\nValidation label distribution:")
display(valid_part[TARGET_COLUMN].value_counts().sort_index().to_frame("count"))

train_dataset = ImageClassificationDataset(train_part, label_col="label_idx", transform=train_transform)
valid_dataset = ImageClassificationDataset(valid_part, label_col="label_idx", transform=valid_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

## 11. Baseline Model Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def build_model(num_classes):
    weights = None
    loaded_pretrained = False
    if USE_PRETRAINED:
        try:
            weights = models.ResNet18_Weights.DEFAULT
        except Exception:
            weights = None

    try:
        model = models.resnet18(weights=weights)
        loaded_pretrained = weights is not None
    except Exception as exc:
        print("Could not load pretrained weights. Falling back to random initialization.")
        print("Reason:", exc)
        model = models.resnet18(weights=None)

    if FREEZE_BACKBONE and loaded_pretrained:
        for param in model.parameters():
            param.requires_grad = False
    elif FREEZE_BACKBONE and not loaded_pretrained:
        print("Backbone will stay trainable because pretrained weights are unavailable.")

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = build_model(num_classes=len(label_to_idx)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

def run_one_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for batch in loader:
        images, targets = batch
        images = images.to(device)
        targets = targets.to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, targets)

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(targets.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    acc = accuracy_score(all_targets, all_preds) if all_targets else 0.0
    return avg_loss, acc

history = []
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_one_epoch(model, train_loader, optimizer)
    valid_loss, valid_acc = run_one_epoch(model, valid_loader, optimizer=None)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
    })
    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"valid_loss={valid_loss:.4f} valid_acc={valid_acc:.4f}"
    )

history_df = pd.DataFrame(history)
display(history_df)

## 12. Validation And Metric Calculation

In [ ]:
model.eval()
valid_preds = []
valid_targets = []

with torch.no_grad():
    for images, targets in valid_loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        valid_preds.extend(preds)
        valid_targets.extend(targets.numpy().tolist())

valid_accuracy = accuracy_score(valid_targets, valid_preds)
print(f"Validation accuracy: {valid_accuracy:.5f}")

target_names = [str(idx_to_label[i]) for i in range(len(idx_to_label))]
print("\nClassification report:")
labels = list(range(len(target_names)))
print(classification_report(
    valid_targets,
    valid_preds,
    labels=labels,
    target_names=target_names,
    zero_division=0,
))

cm = confusion_matrix(valid_targets, valid_preds)
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(len(target_names)), target_names)
plt.yticks(range(len(target_names)), target_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.colorbar()
plt.tight_layout()
plt.show()

## 13. Test Prediction

In [ ]:
test_image_root = Path(TEST_IMAGE_DIR) if TEST_IMAGE_DIR else DATA_DIR
all_images_for_test = list_image_files(test_image_root)

stem_to_image_path = {}
name_to_test_image_path = {}
for path in all_images_for_test:
    stem_to_image_path.setdefault(path.stem, path)
    name_to_test_image_path.setdefault(path.name, path)

test_df = sample_submission.copy()

def map_test_id_to_path(value):
    value = str(value)
    if value in name_to_test_image_path:
        return name_to_test_image_path[value]
    if Path(value).stem in stem_to_image_path:
        return stem_to_image_path[Path(value).stem]
    # Common Kaggle pattern: id has no extension but image file is id + .jpg/.png.
    for ext in IMAGE_EXTENSIONS:
        candidate = value + ext
        if candidate in name_to_test_image_path:
            return name_to_test_image_path[candidate]
    return None

test_df["image_path"] = test_df[ID_COLUMN].map(map_test_id_to_path)
missing_test_images = test_df["image_path"].isna().sum()
print("Test rows:", len(test_df))
print("Test rows without matched image file:", missing_test_images)
if missing_test_images:
    display(test_df.loc[test_df["image_path"].isna()].head())
    raise FileNotFoundError("Some test images could not be matched. Check ID_COLUMN, TEST_IMAGE_DIR, and file extensions.")

test_dataset = ImageClassificationDataset(test_df, label_col=None, transform=valid_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model.eval()
test_pred_indices = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        test_pred_indices.extend(preds)

test_pred_labels = [idx_to_label[idx] for idx in test_pred_indices]
print("Generated predictions:", len(test_pred_labels))
print("Prediction distribution:")
display(pd.Series(test_pred_labels).value_counts().sort_index().to_frame("count"))

## 14. Submission File Generation

In [ ]:
submission = sample_submission.copy()
submission[SUBMISSION_TARGET_COLUMN] = test_pred_labels

# Keep exactly the sample submission columns and row order.
submission = submission[sample_submission.columns]

OUTPUT_PATH = Path(OUTPUT_PATH)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

print("Saved submission to:", OUTPUT_PATH)
print("Submission shape:", submission.shape)
display(submission.head())

if IN_COLAB:
    files.download(str(OUTPUT_PATH))

## 15. Optional Improvement Ideas

- Increase `EPOCHS` after the baseline runs successfully.
- Set `FREEZE_BACKBONE = False` and lower `LEARNING_RATE` to fine-tune the full network.
- Try larger image sizes such as `IMG_SIZE = 256` or `384` if GPU memory allows.
- Use stronger augmentation after verifying it does not hurt validation accuracy.
- Try `torchvision` models such as EfficientNet-B0/B2 or ConvNeXt-Tiny.
- Use k-fold cross-validation if the public/private leaderboard split is unstable.
- Add test-time augmentation by averaging predictions from original and flipped images.
- Review mislabeled or low-confidence validation images manually.

## 16. Debug Checklist

- Does `DATA_DIR` contain extracted files after the extraction cell?
- Do `TRAIN_PATH` and `SAMPLE_SUBMISSION_PATH` point to the intended CSV files?
- Are `IMAGE_COLUMN`, `TARGET_COLUMN`, `ID_COLUMN`, and `SUBMISSION_TARGET_COLUMN` correct?
- Does the notebook report zero missing training images?
- Does the notebook report zero missing test images?
- Does validation use the same metric as the leaderboard, usually accuracy here?
- Does `submission.csv` have the same columns and row count as `sample_submission.csv`?
- Are prediction labels in the original label format, not internal encoded indices?